# T60 AUV 精度训练

训练行为全部来自 `t60_trajectory_precision_v16.json`。物理仿真保持 100 Hz，policy、观测和 8 路动作更新为 25 Hz；融合状态仅保留确定性 50 ms 延迟，不注入传感器噪声；Actor 使用 8 个先前样本形成 320 ms 历史。所有目标姿态严格保持 `roll=pitch=0`，仅让 yaw 跟随水平速度方向；姿态 reward 对 roll、pitch 和 yaw 独立计算并等权汇总，精度半宽为 `2.5°`。Actor 以 `z ~ Normal(μ, σ)` 采样并直接输出 `tanh(z)` 形式的 8 路 `[-1,1]` 推进器命令，训练、评估和 ONNX 使用同一动作语义，再按 `PWM_model = 1500 + 250·command` 映射到 `1250–1750 µs`；纯轴正弦覆盖 `0.1–0.5 m/s` 五档。`precision_v9` reward 以 `0.010` 对八路有界电机指令的平方和收费，并以 `0.010` 对归一化一阶变化的平方均值收费；二阶变化以系数 `2.5` 约束全部确定性有界命令，并以系数 `0.5` 对 T1–T4 四个垂向推进器通道的二阶变化 RMS 额外收费，不做带符号聚合。PPO 使用 Actor `3e-5`、Critic `3e-4` 的固定学习率；解析 KL 超限后只停止本轮 Actor 更新，Critic 继续完成全部 minibatch。当前从头训练 500 个迭代。水动力使用最近一次 CFD 的完整 6×6 响应矩阵（保留物理允许的非对角项），并分别随机化线性阻尼、二次阻尼和附加质量；推进器同时保留逐台增益失配与每个环境唯一的全体弱化系数。三轴纯正弦训练加减速与反向，横向和垂向前进正弦使用显式速度—曲率配对。Notebook 只管理一个前台 worker，最后一个单元格结束、停止或被中断时，训练进程组也会终止。

In [ ]:
from pathlib import Path
import sys

REPO_ROOT = Path.cwd().resolve()
if not (REPO_ROOT / 'simulation/training').is_dir():
    raise RuntimeError('请从 isaac-auv-env 仓库根目录启动 train.ipynb。')
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from simulation.training import build_training_campaign
from simulation.training.campaign import build_train_command, display_command, run_command


## 1. 选择运行

In [ ]:
RECIPE_PATH = REPO_ROOT / 'simulation/training/recipes/t60_trajectory_precision_v16.json'
RUN_NAME = 't60_precision_v16'
SEED = 42
NUM_ENVS = 2048
HEADLESS = True
START_TRAINING = True


## 2. 校验并预览

Actor 为 201 维 `mlp_history_8`（当前 33 维 + 168 维历史）；TensorBoard 的 Scalars 中可搜索 `curriculum/`、`trajectory/`、`tracking/`、`reward/` 和 `domain_randomization/`，核对阶段、速度、曲率、误差、奖励分量和 DR 实际采样统计。

In [ ]:
campaign = build_training_campaign(
    isaaclab_root=Path.home() / 'IsaacLab',
    recipe_path=RECIPE_PATH,
    seed=SEED,
    num_envs=NUM_ENVS,
    run_name=RUN_NAME,
    headless=HEADLESS,
)
train_command = build_train_command(campaign.experiment, campaign.train)
print(f'recipe: {campaign.recipe.name}')
print(f'architecture: {campaign.recipe.mlp_architecture} ({campaign.recipe.architecture.observation_dim} dims)')
print(f'reward: {campaign.recipe.reward_profile}')
print(f'iterations: {campaign.recipe.max_iterations}')
print(display_command(train_command, cwd=campaign.experiment.isaaclab_root))


## 3. 运行训练

这是唯一的训练管理单元格。当前从零开始训练 500 个迭代，不加载任何 checkpoint。中断本单元格会向整个训练进程组发送终止信号，不会留下脱离 notebook 的 worker。

In [ ]:
if not START_TRAINING:
    print('未启动：将 START_TRAINING 改为 True 后重新运行本单元。')
else:
    print(f'TensorBoard: tensorboard --logdir {campaign.experiment.logs_root}')
    run_command(
        train_command,
        cwd=campaign.experiment.isaaclab_root,
        execute=True,
        label='TRAIN',
    )
